In [3]:
import json
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import random

In [ ]:
class SurveyResponseAnalyzer:

    ### for semantic/thematic analysis 

    def __init__(self):
    
        self.burglary_negative_keywords = [
            'afraid', 'scared', 'worried', 'anxious', 'terrified', 'frightened', 'paranoid',
            'unsafe', 'insecure', 'vulnerable', 'threatened', 'helpless', 'nervous',
            'break-in', 'broke in', 'burglar', 'burglary', 'stolen', 'theft', 'robbed',
            'crime', 'criminal', 'dangerous', 'risky', 'sketchy', 'bad neighborhood',
            'high crime', 'increasing crime', 'crime wave', 'getting worse', 'deteriorating'
        ]
        
        self.burglary_positive_keywords = [
            'safe', 'secure', 'protected', 'confident', 'comfortable', 'peaceful',
            'quiet', 'good neighborhood', 'low crime', 'improving', 'better', 'safer',
            'security system', 'alarm', 'cameras', 'locks', 'gated', 'patrol',
            'neighborhood watch', 'community', 'trust', 'reliable'
        ]
        
        self.police_negative_keywords = [
            'slow response', 'never see police', 'no police', 'inadequate', 'useless',
            'don\'t care', 'ineffective', 'lazy', 'corrupt', 'unhelpful', 'rude',
            'understaffed', 'too few officers', 'budget cuts', 'defunded',
            'long wait', 'delayed', 'ignored', 'dismissed', 'poor service',
            'dissatisfied', 'disappointed', 'frustrated', 'angry'
        ]
        
        self.police_positive_keywords = [
            'quick response', 'fast', 'helpful', 'professional', 'courteous',
            'visible', 'patrol', 'community policing', 'accessible', 'reliable',
            'effective', 'sufficient', 'adequate', 'good service', 'satisfied',
            'impressed', 'responsive', 'present', 'active', 'engaged'
        ]
        
        self.general_negative = [
            'terrible', 'awful', 'horrible', 'bad', 'worst', 'hate', 'disgusting',
            'nightmare', 'disaster', 'mess', 'broken', 'failed', 'useless'
        ]
        
        self.general_positive = [
            'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'love',
            'perfect', 'brilliant', 'outstanding', 'impressed', 'satisfied'
        ]


    ### created answers 

    def generate_synthetic_responses(self, num_responses=100):
        
        # for negative responses 
        negative_templates = [
            "I'm really {neg_word} about break-ins in our area. The {crime_word} has been {increase_word} and I never see any police {patrol_word}.",
            "Our neighborhood feels {unsafe_word}. Had a {crime_word} last month and police took {time_word} to respond. Very {dissatisfied_word}.",
            "Crime is getting {worse_word}. I'm {worried_word} to leave my house because the area is becoming {dangerous_word}.",
            "Police presence is {inadequate_word}. There are {too_few_word} officers and response time is {slow_word}.",
            "I feel {vulnerable_word} in my own home. The {crime_word} situation is {terrible_word} and getting worse.",
            "Never see police patrols anymore. It's {scary_word} how {crime_word} has increased in our street.",
            "Break-ins are happening {frequently_word}. Police don't seem to {care_word} about our safety.",
            "I'm {anxious_word} all the time about home security. The police response is {poor_word}.",
            "Our area has become {risky_word}. Too many {burglaries_word} and not enough police {visibility_word}.",
            "Feeling {insecure_word} about leaving home. Crime is {out_of_control_word} and police are {useless_word}."
        ]
        
        # for positive responses
        positive_templates = [
            "I feel very {safe_word} in our neighborhood. The police {patrol_word} regularly and are very {responsive_word}.",
            "Our area is {secure_word} and {peaceful_word}. Great police presence and {quick_word} response times.",
            "Crime has been {decreasing_word} in our area. Police are {active_word} and {professional_word}.",
            "I'm {confident_word} about home security. Good {community_watch_word} and {effective_word} policing.",
            "Police response is {excellent_word}. They arrive {quickly_word} and are very {helpful_word}.",
            "Our neighborhood is {safe_word} and {quiet_word}. Regular police {patrols_word} make me feel {secure_word}.",
            "I trust our local police. They're {reliable_word} and {professional_word} in their service.",
            "Crime rates are {low_word} here. Police are {visible_word} and {engaged_word} with the community.",
            "Feel {comfortable_word} leaving my house. Good {security_word} and {adequate_word} police presence.",
            "Our area is {improving_word}. Police work is {effective_word} and response time is {fast_word}."
        ]
        
        negative_words = {
            'neg_word': ['worried', 'scared', 'afraid', 'anxious', 'terrified'],
            'crime_word': ['crime', 'burglary', 'theft', 'break-ins'],
            'increase_word': ['increasing', 'rising', 'getting worse', 'escalating'],
            'patrol_word': ['on patrol', 'around', 'patrolling'],
            'unsafe_word': ['unsafe', 'insecure', 'dangerous', 'risky'],
            'time_word': ['hours', '2 hours', 'forever', 'ages'],
            'dissatisfied_word': ['dissatisfied', 'disappointed', 'frustrated', 'angry'],
            'worse_word': ['worse', 'terrible', 'awful', 'horrible'],
            'worried_word': ['worried', 'scared', 'afraid', 'reluctant'],
            'dangerous_word': ['dangerous', 'unsafe', 'risky', 'sketchy'],
            'inadequate_word': ['inadequate', 'insufficient', 'poor', 'lacking'],
            'too_few_word': ['too few', 'not enough', 'insufficient'],
            'slow_word': ['slow', 'delayed', 'poor', 'terrible'],
            'vulnerable_word': ['vulnerable', 'exposed', 'helpless', 'defenseless'],
            'terrible_word': ['terrible', 'awful', 'horrible', 'appalling'],
            'scary_word': ['scary', 'frightening', 'alarming', 'disturbing'],
            'frequently_word': ['frequently', 'often', 'regularly', 'constantly'],
            'care_word': ['care', 'bother', 'help', 'respond properly'],
            'anxious_word': ['anxious', 'nervous', 'stressed', 'worried'],
            'poor_word': ['poor', 'terrible', 'inadequate', 'disappointing'],
            'risky_word': ['risky', 'dangerous', 'unsafe', 'hazardous'],
            'burglaries_word': ['burglaries', 'break-ins', 'thefts', 'crimes'],
            'visibility_word': ['visibility', 'presence', 'patrols', 'activity'],
            'insecure_word': ['insecure', 'unsafe', 'vulnerable', 'worried'],
            'out_of_control_word': ['out of control', 'rampant', 'everywhere', 'escalating'],
            'useless_word': ['useless', 'ineffective', 'hopeless', 'inadequate']
        }
        
        positive_words = {
            'safe_word': ['safe', 'secure', 'protected', 'comfortable'],
            'patrol_word': ['patrol', 'monitor', 'watch over', 'guard'],
            'responsive_word': ['responsive', 'quick', 'helpful', 'professional'],
            'secure_word': ['secure', 'safe', 'protected', 'well-guarded'],
            'peaceful_word': ['peaceful', 'quiet', 'calm', 'tranquil'],
            'quick_word': ['quick', 'fast', 'rapid', 'prompt'],
            'decreasing_word': ['decreasing', 'dropping', 'reducing', 'improving'],
            'active_word': ['active', 'engaged', 'involved', 'present'],
            'professional_word': ['professional', 'courteous', 'helpful', 'competent'],
            'confident_word': ['confident', 'assured', 'comfortable', 'relaxed'],
            'community_watch_word': ['community watch', 'neighborhood security', 'local vigilance'],
            'effective_word': ['effective', 'successful', 'efficient', 'good'],
            'excellent_word': ['excellent', 'outstanding', 'great', 'superb'],
            'quickly_word': ['quickly', 'fast', 'promptly', 'immediately'],
            'helpful_word': ['helpful', 'supportive', 'caring', 'attentive'],
            'quiet_word': ['quiet', 'peaceful', 'calm', 'serene'],
            'patrols_word': ['patrols', 'rounds', 'monitoring', 'surveillance'],
            'reliable_word': ['reliable', 'dependable', 'trustworthy', 'consistent'],
            'low_word': ['low', 'minimal', 'rare', 'infrequent'],
            'visible_word': ['visible', 'present', 'around', 'active'],
            'engaged_word': ['engaged', 'involved', 'connected', 'responsive'],
            'comfortable_word': ['comfortable', 'relaxed', 'at ease', 'confident'],
            'security_word': ['security', 'protection', 'safety measures', 'surveillance'],
            'adequate_word': ['adequate', 'sufficient', 'good', 'proper'],
            'improving_word': ['improving', 'getting better', 'enhancing', 'developing'],
            'fast_word': ['fast', 'quick', 'rapid', 'immediate']
        }
        
        responses = []
        
        for i in range(num_responses):
            is_negative = random.random() < 0.6 # random positive to negative 
            
            if is_negative:
                template = random.choice(negative_templates)
                filled_template = template
                for placeholder, word_list in negative_words.items():
                    if '{' + placeholder + '}' in filled_template:
                        filled_template = filled_template.replace('{' + placeholder + '}', random.choice(word_list))
                
                response = {
                    'id': str(i + 1),
                    'text': filled_template,
                    'sentiment_label': 'negative'
                }
            else:
                template = random.choice(positive_templates) ## if the previous doesn't work 
                filled_template = template
                for placeholder, word_list in positive_words.items():
                    if '{' + placeholder + '}' in filled_template:
                        filled_template = filled_template.replace('{' + placeholder + '}', random.choice(word_list))
                
                response = {
                    'id': str(i + 1),
                    'text': filled_template,
                    'sentiment_label': 'positive'
                }
            
            responses.append(response)
        
        return responses

    def load_survey_data(self, filename):
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data
        except FileNotFoundError:
            print(f"File {filename} not found. Creating synthetic survey responses for demonstration.")
            return self.generate_synthetic_responses(100)

    def generate_survey_responses(self, sentiment_label, add_variation=True): # creating responses based on the pre-assigned sentiment 
        
        # baseline 
        responses = { 
            'worry_about_burglary': 3,  # Q1: How worried about break-in
            'burglary_chance_change': 3,  # Q2: Chance compared to year ago
            'confidence_belongings_safe': 3,  # Q3: Confidence belongings are safe
            'police_patrol_frequency': 3,  # Q4: How often see police patrol
            'police_officer_numbers': 3,  # Q5: Number of officers adequate
            'police_response_confidence': 3,  # Q6: Confidence in quick police response
            'police_response_satisfaction': 3  # Q7: Satisfaction with police response
        }
        
        # adjustment based on sentiment label
        if sentiment_label == 'negative':
            strength = random.randint(2, 3)  # Random strength for variation
            responses['worry_about_burglary'] = max(1, 3 - strength)  # More worried
            responses['burglary_chance_change'] = max(1, 3 - strength)  # Think chance is higher
            responses['confidence_belongings_safe'] = max(1, 3 - strength)  # Less confident
            responses['police_patrol_frequency'] = max(1, 3 - strength)  # See police less often
            responses['police_officer_numbers'] = max(1, 3 - strength)  # Too few officers
            responses['police_response_confidence'] = max(1, 3 - strength)  # Less confident in response
            responses['police_response_satisfaction'] = max(1, 3 - strength)  # Less satisfied
            
        elif sentiment_label == 'positive':
            strength = random.randint(1, 2)  # Random strength for variation
            responses['worry_about_burglary'] = min(5, 3 + strength)  # Less worried
            responses['burglary_chance_change'] = min(5, 3 + strength)  # Think chance is lower
            responses['confidence_belongings_safe'] = min(5, 3 + strength)  # More confident
            responses['police_patrol_frequency'] = min(5, 3 + strength)  # See police more often
            responses['police_officer_numbers'] = min(5, 3 + strength)  # Adequate/more than enough officers
            responses['police_response_confidence'] = min(5, 3 + strength)  # More confident in response
            responses['police_response_satisfaction'] = min(5, 3 + strength)  # More satisfied
        
        # Add realistic random variation
        if add_variation:
            for key in responses:
                variation = random.choice([-1, 0, 1]) * 0.5  # random adjustment
                responses[key] = max(1, min(5, responses[key] + variation))
                responses[key] = round(responses[key])
        
        return responses

    def create_edge_case_lsoas(self): # one really good one really bad LSOA
        edge_cases = []
        
        # very good LSOA
        good_lsoa = {
            'post_id': 'EDGE_GOOD',
            'lsoa_id': 'LSOA_EXCELLENT_001',
            'text': 'Our area is absolutely fantastic. Crime has dropped significantly and police response is excellent. I feel completely safe and confident about home security. Regular patrols and professional service.',
            'sentiment_label': 'positive',
            'case_type': 'very_good'
        }
        edge_cases.append(good_lsoa)
        
        # very bad LSOA
        bad_lsoa = {
            'post_id': 'EDGE_BAD',
            'lsoa_id': 'LSOA_TERRIBLE_001',
            'text': 'This area is absolutely terrible. Crime is out of control with constant break-ins. Police never patrol and response times are awful. I\'m terrified to leave my house and feel completely unsafe.',
            'sentiment_label': 'negative',
            'case_type': 'very_bad'
        }
        edge_cases.append(bad_lsoa)
        
        return edge_cases

    def process_all_responses(self, filename='posts-dataset.json'): # here generate actual answers
        survey_data = self.load_survey_data(filename)
        
        results = []
        post_counter = 1
        
        for response in survey_data:
            lsoa_id = f"LSOA_{post_counter:04d}"
            
            if 'sentiment_label' in response:
                sentiment_label = response['sentiment_label']
            else:
                # if no pre-label
                sentiment_analysis = self.analyze_sentiment(response.get('text', ''))
                sentiment_label = sentiment_analysis['overall_sentiment']
            
            # generate survey responses
            survey_responses = self.generate_survey_responses(sentiment_label)
            
            # records
            result = {
                'post_id': post_counter,
                'lsoa_id': lsoa_id,
                'open_answer': response.get('text', ''),
                'sentiment_label': sentiment_label,
                **survey_responses
            }
            
            results.append(result)
            post_counter += 1
        
        return results

    def analyze_sentiment(self, text): # fallback 
        text_lower = text.lower()
        
        burglary_negative_count = sum(1 for word in self.burglary_negative_keywords if word in text_lower)
        burglary_positive_count = sum(1 for word in self.burglary_positive_keywords if word in text_lower)
        police_negative_count = sum(1 for word in self.police_negative_keywords if word in text_lower)
        police_positive_count = sum(1 for word in self.police_positive_keywords if word in text_lower)
        general_negative_count = sum(1 for word in self.general_negative if word in text_lower)
        general_positive_count = sum(1 for word in self.general_positive if word in text_lower)
        
        total_negative = burglary_negative_count + police_negative_count + general_negative_count
        total_positive = burglary_positive_count + police_positive_count + general_positive_count
        
        if total_negative > total_positive:
            overall_sentiment = 'negative'
        elif total_positive > total_negative:
            overall_sentiment = 'positive'
        else:
            overall_sentiment = 'neutral'
        
        return {'overall_sentiment': overall_sentiment}

    def create_csv_dataset(self, filename='posts-dataset.json', output_filename='survey_responses.csv', num_synthetic=200):
    
        # regular responses
        regular_results = self.process_all_responses(filename)
        
        # synthetic responses 
        if len(regular_results) < 50: # if too little official ones
            print(f"Adding {num_synthetic} synthetic responses...")
            synthetic_data = self.generate_synthetic_responses(num_synthetic)
            
            for i, response in enumerate(synthetic_data):
                survey_responses = self.generate_survey_responses(response['sentiment_label'])
                
                result = {
                    'post_id': len(regular_results) + i + 1,
                    'lsoa_id': f"LSOA_{len(regular_results) + i + 1:04d}",
                    'open_answer': response['text'],
                    'sentiment_label': response['sentiment_label'],
                    **survey_responses
                }
                regular_results.append(result)
        
        # edge cases
        edge_cases = self.create_edge_case_lsoas()
        edge_results = []
        
        for case in edge_cases:
            survey_responses = self.generate_survey_responses(case['sentiment_label'], add_variation=False)
            
            if case['case_type'] == 'very_good':
                survey_responses = {k: 5 for k in survey_responses.keys()}  # All maximum positive
                survey_responses['worry_about_burglary'] = 5  # Not worried at all
                survey_responses['burglary_chance_change'] = 5  # Much lower chance
            elif case['case_type'] == 'very_bad':
                survey_responses = {k: 1 for k in survey_responses.keys()}  # All maximum negative
                survey_responses['worry_about_burglary'] = 1  # Very worried
                survey_responses['burglary_chance_change'] = 1  # Much higher chance
            
            result = {
                'post_id': case['post_id'],
                'lsoa_id': case['lsoa_id'],
                'open_answer': case['text'],
                'sentiment_label': case['sentiment_label'],
                **survey_responses
            }
            
            edge_results.append(result)
    
        all_results = regular_results + edge_results
        
        df = pd.DataFrame(all_results)
        
        column_order = [
            'post_id', 'lsoa_id', 'sentiment_label',
            'worry_about_burglary', 'burglary_chance_change', 'confidence_belongings_safe',
            'police_patrol_frequency', 'police_officer_numbers', 
            'police_response_confidence', 'police_response_satisfaction',
            'open_answer'
        ]
        
        df = df[column_order]

        df.to_csv(output_filename, index=False)
        print(f"Dataset saved to {output_filename}")
        print(f"Total records: {len(df)}")
        print(f"Regular responses: {len(regular_results)}")
        print(f"Edge cases: {len(edge_results)}")
        
        return df

if __name__ == "__main__":
    analyzer = SurveyResponseAnalyzer()
    
    df = analyzer.create_csv_dataset('posts-dataset.json', 'survey_responses.csv')
    
    print("\n=== DATASET SUMMARY ===")
    print(f"Shape: {df.shape}")
    print(f"\nSentiment Distribution:")
    print(df['sentiment_label'].value_counts())
    print(f"\nSample responses:")
    print(df[['post_id', 'lsoa_id', 'sentiment_label', 'worry_about_burglary', 'police_response_satisfaction']].head(10))
    
    print(f"\nEdge Cases:")
    edge_cases = df[df['lsoa_id'].str.contains('EXCELLENT|TERRIBLE')]
    if not edge_cases.empty:
        print(edge_cases[['lsoa_id', 'sentiment_label', 'worry_about_burglary', 'police_response_satisfaction']])